# nb/05: the call-day console on a Colab GPU (backup to the Space)

A backup for when the Hugging Face Space is down. Runs the same console the Space runs, from
`cultureqc_console_bundle.zip` (made on the Mac by `scripts/make_console_bundle.py`: the synced
`deploy/hf-space-demo/` plus the model weights), so nothing is downloaded from Hugging Face.
Only PyPI is used, for the Python packages.

Follows the Drive I/O rule: the zip is copied from Drive to Colab's local disk and everything runs
from `/content`. One small result file goes back to Drive.

1. Runtime > Change runtime type > **T4 GPU** (or better).
2. Upload `cultureqc_console_bundle.zip` to `MyDrive/cultureqc/` and wait for the upload to finish (step 2 also finds it elsewhere in MyDrive; the newest copy wins).
3. Run the cells in order. Step 4 is the dry run; step 5 opens the console.

In [ ]:
# 1. GPU check
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU: switch the runtime to T4 GPU')

In [ ]:
# 2. Drive -> local disk, unzip, check every file against the bundle's manifest
import glob, hashlib, json, os, shutil, zipfile
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/cultureqc'
LOCAL_ZIP = '/content/cultureqc_console_bundle.zip'
# The expected place first, then anywhere under MyDrive up to 3 folders deep (a browser upload
# can land in MyDrive itself, and a re-upload can be renamed 'cultureqc_console_bundle (1).zip').
# The newest match wins; the manifest check below says which commit it is.
found = glob.glob(f'{DRIVE_DIR}/cultureqc_console_bundle*.zip') or [
    q for d in range(4)
    for q in glob.glob('/content/drive/MyDrive/' + '*/' * d + 'cultureqc_console_bundle*.zip')]
if not found:
    raise FileNotFoundError(
        'cultureqc_console_bundle*.zip is not on Drive (looked in MyDrive and 3 folders down). '
        'Upload it from the Mac (~/Desktop/projs/cultureqc_console_bundle.zip, 1.42 GB) and wait until '
        'the upload finishes; then Runtime > Restart session and run again, or run '
        'drive.flush_and_unmount() and this cell again. Contents of DRIVE_DIR: '
        + str(sorted(os.listdir(DRIVE_DIR)) if os.path.isdir(DRIVE_DIR) else 'folder missing'))
src_zip = max(found, key=os.path.getmtime)
print('bundle on Drive:', src_zip, f'({os.path.getsize(src_zip) / 1e9:.2f} GB)',
      '| other copies:', [q for q in found if q != src_zip] or 'none')
shutil.copy(src_zip, LOCAL_ZIP)
shutil.rmtree('/content/console', ignore_errors=True)
with zipfile.ZipFile(LOCAL_ZIP) as z:
    z.extractall('/content')
os.remove(LOCAL_ZIP)

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()

man = json.load(open('/content/console/BUNDLE.json'))
bad = [p for p, h in man['files'].items() if sha256('/content/' + p) != h]
print('bundle commit', man['git_commit'][:10], '(dirty)' if man['git_dirty'] else '',
      '|', len(man['files']), 'files | mismatched:', bad or 'none')
assert not bad

In [ ]:
# 3. Packages from PyPI. Keeps Colab's own CUDA torch (the Space's torch pin is for ZeroGPU).
skip = ('torch', 'torchvision', 'spaces')
reqs = [l for l in open('/content/console/requirements.txt').read().splitlines()
        if l.strip() and not l.lstrip().startswith('#') and l.split('>=')[0].split('==')[0].strip() not in skip]
open('/content/requirements_colab.txt', 'w').write('\n'.join(reqs) + '\ngradio==6.26.0\n')
!pip install -q -r /content/requirements_colab.txt

In [ ]:
# 4. Dry run: re-run all 7 precomputed examples live, compare with the stored outputs, time each stage.
#    Offline flags: the weights come from the bundle, never from Hugging Face.
import subprocess
ENV = dict(os.environ,
           HF_HUB_CACHE='/content/console/weights/hf/hub', HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1',
           CELLPOSE_LOCAL_MODELS_PATH='/content/console/weights/cellpose', GRADIO_ANALYTICS_ENABLED='False')
OUT = '/content/live_latency_colab_gpu.md'
r = subprocess.run(['python', '-m', 'demo.selfcheck', '--n', '7', '--out', OUT],
                   cwd='/content/console', env=ENV, capture_output=True, text=True)
print(r.stdout[-8000:])
if r.returncode:
    print(r.stderr[-4000:])
else:
    shutil.copy(OUT, f'{DRIVE_DIR}/live_latency_colab_gpu.md')   # small result back to Drive
    print('copied to', f'{DRIVE_DIR}/live_latency_colab_gpu.md')

In [ ]:
# 5. Start the console and open it through Colab's own proxy (works in this browser, signed in to this
#    Google account; share this browser window on the call). The log is per boot, as on the Space.
import time, urllib.request
log = open('/content/console.log', 'w')
proc = subprocess.Popen(['python', '-u', 'console.py'], cwd='/content/console',
                        env=dict(ENV, GRADIO_SERVER_PORT='7860'), stdout=log, stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        urllib.request.urlopen('http://127.0.0.1:7860/config', timeout=2)
        break
    except Exception:
        time.sleep(2)
print(''.join(l for l in open('/content/console.log') if l.startswith(('cultureQC', '* Running'))))
from google.colab import output
# serve_kernel_port_as_window is deprecated (Colab warns it may stop working), so: the proxy URL
# for a full browser tab (use this on the call), and the same console inline below as a fallback.
url = output.eval_js('google.colab.kernel.proxyPort(7860, {"cache": false})')
print('Console, full tab (works only in this browser, signed in to this Google account):', url)
output.serve_kernel_port_as_iframe(7860, height=900)

Optional: a public `gradio.live` link instead (for someone else to open). It goes through Gradio's share
servers, which Hugging Face runs, so it may not work during the same outage; the Colab window above does
not depend on it. Stop the console first (`proc.terminate()`), then start it again with
`env=dict(ENV, GRADIO_SERVER_PORT='7860', CULTUREQC_SHARE='1')` and read the link from `/content/console.log`.

After the dry run, download `live_latency_colab_gpu.md` from Drive and commit it as
`results/live_latency_colab_gpu.md`; quote its numbers, not a guess.

In [ ]:
# Stop the console when done.
proc.terminate()